# Financial Econometrics I

#### Modeling Volatility II - Seminar

by Lukas Vacha and Josef Kurka

#### Seminar 5: Summer Semester 2025/2026

The data for seminars can be downloaded on [Google drive](https://drive.google.com/drive/folders/11x60HJIClqGWNUEqIT9gynkgJumnFxR0?usp=share_link)

For this seminar you will need `data/dax.csv` and `data/AAPL.csv`.

___


In [ ]:
if (!require(rugarch)) install.packages('rugarch')
if (!require(repr)) install.packages('repr')
if (!require(tseries)) install.packages('tseries')
if (!require(forecast)) install.packages('forecast')
if (!require(aTSA)) install.packages('aTSA')
if (!require(arfima)) install.packages('arfima')
if (!require(longmemo)) install.packages('longmemo')
if (!require(ggplot2)) install.packages('ggplot2')
if (!require(readr)) install.packages('readr')

In [ ]:
library(repr)
library(rugarch)
library(tseries)
library(forecast)
library(aTSA)
library(arfima)
library(longmemo)
library(ggplot2)
library(readr)
library(quantmod)

options(repr.plot.width = 12, repr.plot.height = 8)

### GARCH-in-mean model

In both theory and practice, investors require a risk-premium for stocks with higher volatility. Therefore, volatility may be directly included in the return generating mechanism of particular security. Such relationship is then modelled by GARCH-M model, e.g. GARCH(1, 1)-M model is formalized as:

$$ r_t = \mu + c \sigma_t + a_t $$
$$   a_t = \sigma_t \epsilon_t $$
$$  \sigma_t^2 = \omega + \alpha_1 a_{t-1}^2 + \beta_1 \sigma_{t-1}^2$$
   
Let's try simulating such process.

In [ ]:
set.seed(660)
## GARCH(1,1)-M random number generation
## r[t] = mu + c * sigma^2[t] + a[t]
## a[t] = epsilon[t] * sigma[t]
## sigma^2[t] = omega + alpha * a[t-1]^2 + beta * sigma^2[t-1]
nT <- 500; burnin = 500; T = burnin + nT
a <- numeric(T)
sigma <- numeric(T)
epsilon <- numeric(T)
r <- numeric(T)
#step1: generate epsilon~N(0,1)
epsilon <- rnorm(T)
mu <- 0.1
c <- 0.4
omega <- 10 ^ (-6)
alpha <- 0.3
beta <- 0.5
sigma[1] <- (omega / (1 - alpha - beta))^2
a[1] <- epsilon[1] * sqrt(sigma[1])
r[1] <- mu + c * sigma[1] + a[1]
for(t in 2:T){
sigma[t] = omega + alpha * (a[t - 1])^2 + beta * sigma[t - 1]
a[t] = epsilon[t] * sqrt(sigma[t])
r[t] = mu + c * sqrt(sigma[t]) + a[t]
}
# Remove burn-in period
data = r[burnin + 1:nT]
var = sigma[burnin + 1:nT]
# Plot data(y) and vol(h)
par(mfrow = c(1, 2))
ts.plot(data, ylab = NA, main = 'Simulated returns')
ts.plot(sqrt(var), ylab = NA, main = 'Simulated volatility')

In [ ]:
spec <- ugarchspec(variance.model=list(model = "sGARCH", garchOrder = c(1, 1)),
    mean.model = list(armaOrder=c(0, 0), include.mean = TRUE,
    archm = TRUE, archpow = 1))

fit <- ugarchfit(spec, data)

fit

Note that archm coefficient represents c in the mean equation.

## GARCH-in-mean

Here is an example of GARCH-in-mean on real-world data, specifically excess yields of 6 months treasury bills. 

In [ ]:
sixM <- read.csv("data/TB6MS.csv")
threeM <- read.csv("data/TB3MS.csv")

sixM[, 1] <- as.Date(as.character(sixM[, 1]))
threeM[, 1] <- as.Date(as.character(threeM[, 1]))

sixM <- sixM[sixM[, 1] >= as.Date('1960-01-01') & sixM[, 1] < as.Date('1984-05-01') & 
            as.numeric(substr(sixM[, 1], 6, 7)) %% 3 == 1,]
threeM <- threeM[threeM[, 1] >= as.Date('1960-01-01') & threeM[, 1] < as.Date('1984-08-01') & 
            as.numeric(substr(threeM[, 1], 6, 7)) %% 3 == 1,]
sixM[, 2] <- sixM[, 2]/100
threeM[, 2] <- threeM[, 2]/100

head(sixM)
head(threeM)

excess <- 2 * sixM[, 2] - threeM[1:(length(threeM[, 2]) - 1), 2] - threeM[2:length(threeM[, 2]), 2]

plot.ts(excess)

First fit a model including just mean, test the residuals for ARCH effects.

In [ ]:
mu <- arima(excess, order = c(0, 0, 0))
mu
arch.test(mu)

Excess yields are significantly higher than zero. We also reject homoscedasticity at the lowest lags. Fit a GARCH(1, 1) model with volatility included in the mean equation, i.e.

$$ y_t = \mu + c \sigma_{t-1} + \epsilon_t $$
$$ a_t = \sigma_t \epsilon_t $$
$$\sigma_t^2 = \omega + \alpha_1 a_{t-1}^2 + \beta_1 \sigma_{t-1}^2$$

In [ ]:
spec <- ugarchspec(variance.model=list(model = "sGARCH", garchOrder = c(1, 1)),
    mean.model = list(armaOrder=c(0, 0), include.mean = TRUE, archm = TRUE, archpow = 1))

fit <- ugarchfit(spec, excess)

fit

### Data import

In the rest of the seminar, we will work with the DAX index and Apple daily prices. Load and prepare the data.


In [ ]:
# DAX
dax <- read.csv('data/dax.csv')
dax[, 'Date'] <- as.Date(as.character(dax[, 'Date']))

# Apple
aapl <- read.csv('data/AAPL.csv')
aapl[, 'Date'] <- as.Date(as.character(aapl[, 'Date']))

head(dax)
head(aapl)


## DAX: returns and volatility clustering

We begin with the DAX series used in the original forecasting seminar. Compute daily log returns and inspect the series.


In [ ]:
rdax <- dax
rdax[, 2] <- c(diff(log(rdax[, 2])), NA)
rdax <- rdax[-length(rdax[, 1]), ]
colnames(rdax)[2] <- 'return'

par(mfrow = c(1, 2))
plot(dax[, c('Date', names(dax)[2])], type = 'l', main = 'DAX prices', ylab = NA, xlab = NA)
plot(rdax[, c('Date', 'return')], type = 'l', main = 'DAX log returns', ylab = NA, xlab = NA)


In [ ]:
par(mfrow = c(1, 2))
acf(rdax[, 'return'], main = 'ACF of returns')
acf(rdax[, 'return']^2, main = 'ACF of squared returns')


The returns themselves are close to serially uncorrelated, but squared returns display substantial autocorrelation. Therefore a pure GARCH(1,1) specification seems reasonable.

In [ ]:
daxspec <- ugarchspec(mean.model = list(armaOrder=c(0, 0)), variance.model = list(model = "sGARCH", 
                      garchOrder = c(1, 1)))
daxfit <- ugarchfit(daxspec, rdax[, 2])
daxfit

### Forecasting with GARCH and EWMA

Try forecasting the volatility series based on the GARCH model we fitted. Begin with static forecast, i.e. each period volatility is estimated using initial parameters, next period it is reestimated with the same set of parameters except volatility, which is updated based on estimate from previous period. Use forecasting horizon of 250 observations

In [ ]:
horizon <- 250
l <- length(rdax[, 2])
daxfit <- ugarchfit(daxspec, rdax[, 2], out.sample=250)

FGar <- ugarchforecast(daxfit, n.ahead = horizon)
UncVol <- sqrt(uncvariance(daxfit))

plot.ts(daxfit@fit$sigma, ylab = NA, xlim = c(0, l), main = 'Volatility')
lines(rep(UncVol, l + horizon), col = 'grey80')
lines(c(rep(NA, l - horizon - 1), FGar@forecast$sigma), col = 'darkorange')
legend("topleft", ncol = 3, legend = c('GARCH volatility', 'Unconditional Volatility', 'Static forecast'),
      col = c('black', 'grey80', 'darkorange'), lwd = 2, box.col = 'white', cex = 0.85)


In [ ]:
UncVol <- sqrt(uncvariance(daxfit))

FGar <- ugarchforecast(daxfit, n.ahead = horizon)
FDyn <- ugarchforecast(daxfit, n.ahead = 1, n.roll = horizon)

plot.ts(daxfit@fit$sigma, ylab = NA, xlim = c(0, l), main = 'Volatility')
lines(rep(UncVol, l + horizon), col = 'grey80')
lines(c(rep(NA, l - horizon - 1), FGar@forecast$sigma), col = 'darkorange')
lines(c(rep(NA, l - horizon - 1), FDyn@forecast$sigma), col = 'steelblue')
legend("topleft", ncol = 4, legend = c('GARCH volatility', 'Unconditional Volatility', 'Static forecast', 
      'Dynamic forecast'), col = c('black', 'grey80', 'darkorange', 'steelblue'), lwd = 2, 
       box.col = 'white', cex = 0.65, yjust = 0)

Static GARCH forecasts converge to the unconditional volatility

$$\sqrt{\frac{\omega}{1 - \alpha_1 - \beta_1}},$$

while dynamic forecasts react to new information arriving during the out-of-sample period.

In [ ]:
sigmas <- vector()
daxfit <- ugarchfit(daxspec, rdax[, 2], out.sample = horizon)
omega <- coef(daxfit)[2]
alpha <- coef(daxfit)[3]
beta <- coef(daxfit)[4]
L <- length(residuals(daxfit))
sigmas[1] <- sqrt(omega + alpha * residuals(daxfit)[L] ^ 2 + 
                beta * sigma(daxfit)[L] ^ 2)
for (t in 2:horizon){
sigmas[t] <- sqrt(omega + alpha * sigmas[t - 1] ^ 2 + 
                beta * sigmas[t - 1] ^ 2)    
}

In [ ]:
sigmad <- vector()
daxfit <- ugarchfit(daxspec, rdax[, 2], out.sample = horizon)
omega <- coef(daxfit)[2]
alpha <- coef(daxfit)[3]
beta <- coef(daxfit)[4]
L <- length(residuals(daxfit))
sigmad[1] <- sqrt(omega + alpha * residuals(daxfit)[L] ^ 2 + 
                beta * sigma(daxfit)[L] ^ 2)
for (t in 2:100){
daxfit <- ugarchfit(daxspec, rdax[, 2], out.sample = horizon - t + 1) 
L <- length(residuals(daxfit))
omega <- coef(daxfit)[2]
alpha <- coef(daxfit)[3]
beta <- coef(daxfit)[4]
sigmad[t] <- sqrt(omega + alpha * residuals(daxfit)[L] ^ 2 + 
                beta * sigma(daxfit)[L] ^ 2)    
}


In [ ]:
par(mfrow = c(1, 2))
plot.ts(sigmas)
lines(FGar@forecast$sigma, col = 'red')
legend("top", ncol = 2, legend = c('Function', 'Manual'), col = c('black', 'red'), lwd = 2, 
       box.col = 'white', cex = 0.6, yjust = 0)
    
plot.ts(sigmad)
lines(FDyn@forecast$sigma[1:100], col = 'red')
legend("top", ncol = 2, legend = c('Function', 'Manual'), col = c('black', 'red'), lwd = 2, 
       box.col = 'white', cex = 0.6, yjust = 0)

Compare the GARCH fit with the EWMA recursion

$$\sigma^2_{t+1|t} = (1-\lambda) r_t^2 + \lambda \sigma^2_{t|t-1},$$

using the standard RiskMetrics value $\lambda = 0.94$.

In [ ]:
daxfit <- ugarchfit(daxspec, rdax[, 2])
ewma_var <- rep(var(rdax[, 'return']), nrow(rdax))
lambda <- 0.94
for (t in 2:nrow(rdax)) {
    ewma_var[t] <- lambda * ewma_var[t - 1] + (1 - lambda) * rdax[t - 1, 'return']^2
}

plot(rdax[, 'Date'], sqrt(ewma_var), type = 'l', col = 'darkorange',
     main = 'EWMA versus GARCH fitted volatility', xlab = NA, ylab = 'volatility')
lines(rdax[, 'Date'], sigma(daxfit), col = 'steelblue')
legend('topleft', legend = c('EWMA', 'GARCH(1,1)'), col = c('darkorange', 'steelblue'), lwd = 2)


### Global financial crisis forecast

Restrict the DAX sample to the crisis year 2008 and repeat the forecasting comparison. Does the gap between static and dynamic forecasting become more visible when volatility changes rapidly?


In [ ]:
rdax8 <- rdax[rdax[ ,1] >= as.Date("2008-01-01") & rdax[ ,1] <= as.Date("2008-12-21"), ]

horizon8 <- 65
l8 <- length(rdax8[, 2])

daxfit8 <- ugarchfit(daxspec, rdax8[, 2], out.sample = horizon8)
daxfit9 <- ugarchfit(daxspec, rdax8[, 2])

UncVol8 <- sqrt(uncvariance(daxfit8))

FGar8 <- ugarchforecast(daxfit8, n.ahead = horizon8)
FDyn8 <- ugarchforecast(daxfit8, n.ahead = 1, n.roll = horizon8)

plot.ts(daxfit9@fit$sigma, ylab = NA, xlim = c(0, l8), main = 'DAX 2008: static versus dynamic forecast')
lines(rep(UncVol8, l8), col = 'grey80')
lines(c(rep(NA, l8 - horizon8), FGar8@forecast$sigma), col = 'darkorange')
lines(c(rep(NA, l8 - horizon8), FDyn8@forecast$sigma), col = 'steelblue')
legend("top", ncol = 4, legend = c('GARCH volatility', 'Unconditional Volatility', 'Static forecast', 
      'Dynamic forecast'), col = c('black', 'grey80', 'darkorange', 'steelblue'), lwd = 2, 
       box.col = 'white', cex = 0.6, yjust = 0)

## GARCH with Student's t innovations

The lecture stresses that financial returns are typically fat-tailed, so the Gaussian conditional distribution is often too restrictive. We therefore revisit Apple returns, but now work with **adjusted closing prices**, since raw closing prices are distorted by stock splits.


In [ ]:
rapl <- aapl[, c('Date', 'Adj.Close')]
rapl[, 2] <- c(diff(log(rapl[, 2])), NA)
rapl <- rapl[-length(rapl[, 1]), ]
colnames(rapl)[2] <- 'return'

rapl_nadj <- aapl[, c('Date', 'Close')]
rapl_nadj[, 2] <- c(diff(log(rapl_nadj[, 2])), NA)
rapl_nadj <- rapl_nadj[-length(rapl[, 1]), ]
colnames(rapl_nadj)[2] <- 'return'

par(mfrow = c(1, 2))
plot(aapl[, c('Date', 'Close')], type = 'l', main = 'Apple close', ylab = NA, xlab = NA)
plot(aapl[, c('Date', 'Adj.Close')], type = 'l', main = 'Apple adjusted close', ylab = NA, xlab = NA)

plot(rapl[, c('Date', 'return')], type = 'l', main = 'Apple adjusted log returns', ylab = NA, xlab = NA)
plot(rapl_nadj[, c('Date', 'return')], type = 'l', main = 'Apple close log returns', ylab = NA, xlab = NA)


In [ ]:
rapl <- rapl[rapl[, 'Date'] >= as.Date('2000-01-01'), ]
par(mfrow = c(1, 2))
acf(rapl[, 2], main = "")
pacf(rapl[, 2], main = "")

In [ ]:
aaplspec <- ugarchspec(mean.model = list(armaOrder=c(0, 0)), variance.model = list(model = "sGARCH", 
                      garchOrder = c(1, 1)))
aaplfit <- ugarchfit(aaplspec, rapl[, 2])
aaplfit

In [ ]:
jarque.bera.test(residuals(aaplfit) / sigma(aaplfit))

par(mfrow = c(1, 2))

hist(residuals(aaplfit)/ sigma(aaplfit), breaks = 30, main ='Histogram', cex.main = 0.8, cex.lab = 0.8, xlab = NA,
    cex.axis = 0.8)
box()

qqnorm(residuals(aaplfit)/ sigma(aaplfit), cex.main = 0.8, cex.lab = 0.8, cex.axis = 0.8) 
qqline(residuals(aaplfit)/ sigma(aaplfit), lwd = 2)

The residuals do not seem to be normally distributed, fit the same model, but with residuals conditionally having Student's t-distribution.

In [ ]:
aaplspec2 <- ugarchspec(mean.model = list(armaOrder=c(0, 0)), variance.model = list(model = "sGARCH", 
                      garchOrder = c(1, 1)), distribution.model = 'std')
aaplfit2 <- ugarchfit(aaplspec2, rapl[, 2])
aaplfit2

In [ ]:
aaplspec_norm <- ugarchspec(mean.model = list(armaOrder = c(0, 0)),
                            variance.model = list(model = 'sGARCH', garchOrder = c(1, 1)))
aaplspec_t <- ugarchspec(mean.model = list(armaOrder = c(0, 0)),
                         variance.model = list(model = 'sGARCH', garchOrder = c(1, 1)),
                         distribution.model = 'std')

aaplfit_norm <- ugarchfit(aaplspec_norm, rapl[, 'return'])
aaplfit_t <- ugarchfit(aaplspec_t, rapl[, 'return'])

IC <- do.call(cbind, list(
  "normal"   = infocriteria(aaplfit_norm),
  "Student's t"   = infocriteria(aaplfit_t)
))
colnames(IC) <- c('normal', 'Student\'s t')
IC

In [ ]:
par(mfrow = c(1, 2))
qqnorm(residuals(aaplfit_norm, standardize = TRUE), main = 'Normal Q-Q plot')
qqline(residuals(aaplfit_norm, standardize = TRUE), lwd = 2)

nu <- round(coef(aaplfit_t)['shape'], 2)
qqplot(rt(1000, df = nu), as.numeric(residuals(aaplfit_t, standardize = TRUE)),
       xlab = 'Theoretical quantiles', ylab = 'Sample quantiles',
       main = paste('Student t Q-Q plot, df =', nu))
qqline(as.numeric(residuals(aaplfit_t, standardize = TRUE)), lwd = 2)


## Asymmetric GARCH models

Model the asymmetric reaction of volatility to shocks of different sign.

1. symmetric GARCH(1, 1) with Student's t innovations,
2. EGARCH(1, 1) with Student's t innovations,
3. GJR-GARCH(1, 1) with Student's t innovations.

### EGARCH(1 ,1)

Rugarch package estimates EGARCH(1, 1) as

$$log(\sigma_t^2) = \omega + \{ \alpha_1 z_{t-1} + \gamma_1 (\lvert z_{t-1} \rvert - E \lvert z_{t-1}\rvert ) \} + \beta_1 log( \sigma_{t-1}^2), $$ 

where $z_t$ is standardized innovation, log is the natural logarithm.

The coefficient $\alpha_1$ captures the sign effect, the coefficient $\gamma_1$ captures the size effect.

### GJR-GARCH(1, 1)

GJR-GARCH in the rugarch package is estimated as follows

$$ \sigma_t^2 = \omega + (\alpha_1 \epsilon_{t-1}^2 + \gamma_1 I_{t-1} \epsilon_{t-1}^2) + \beta_1 \sigma_{t-1}^2,$$

where $I_t = 1$ if $\epsilon_t \leq 0$, and $0$ otherwise. Gamma coefficient captures the leverage term. The ARCH effect is $\alpha_1$ for $\epsilon_{t-1} > 0$, $\alpha_1 + \gamma_1$ for $\epsilon_{t-1} > 0$, i.e. if $\gamma > 0$ negative volatility has larger effect and vice versa.

In [ ]:
rdax11 <- rdax[rdax[, 'Date'] >= as.Date('2011-01-01') & rdax[, 'Date'] < as.Date('2012-01-01'), ]

spec_standard <- ugarchspec(mean.model = list(armaOrder = c(0, 0)),
                     variance.model = list(model = 'sGARCH', garchOrder = c(1, 1)),
                     distribution.model = 'std')
spec_egarch <- ugarchspec(mean.model = list(armaOrder = c(0, 0)),
                     variance.model = list(model = 'eGARCH', garchOrder = c(1, 1)),
                     distribution.model = 'std')
spec_gjr <- ugarchspec(mean.model = list(armaOrder = c(0, 0)),
                       variance.model = list(model = 'gjrGARCH', garchOrder = c(1, 1)),
                       distribution.model = 'std')

fit_standard <- ugarchfit(spec_standard, rdax11[, 'return'])
fit_egarch <- ugarchfit(spec_egarch, rdax11[, 'return'])
fit_gjr <- ugarchfit(spec_gjr, rdax11[, 'return'])


In [ ]:
color <- c('gray40', 'darkorange', 'steelblue')
plot(fit_standard@fit$sigma, type = 'l', ylab = 'Volatility', main = 'GARCH models comparison', ylim = c(0.005, 0.05), 
     col = color[1])
lines(fit_egarch@fit$sigma, col = color[2])
lines(fit_gjr@fit$sigma, col = color[3])

legend("top", ncol = 3, legend = c('GARCH(1, 1)', 'eGARCH(1, 1)', 'GJR-GARCH(1, 1)'), col = color, lwd = 2, 
       box.col = 'white', cex = .9)

#### EGARCH and sensitivity to positive and negative shocks.

As mentioned above, EGARCH allows to capture asymmetric reaction to positive and negative shocks. Sensitivity to positive and negative shocks is captured by different slope for positive and negative shocks. From

$$log(\sigma_t^2) = \omega + \{ \alpha_1 z_{t-1} + \gamma_1 (\lvert z_{t-1} \rvert - E \lvert z_{t-1}\rvert ) \} + \beta_1 log( \sigma_{t-1}^2), $$ 

we see effect is $(\alpha_1 + \gamma_1) * z_t$ when $z_t > 0$, $(\alpha_1 - \gamma_1) * z_t$ when $z_t < 0$. In the above estimated EGARCH(1, 1) with normal residuals, the sensitivity is $(\alpha_1 - \gamma_1) * z_t \approx 0.33$ for $z_t = -1$, and $(\alpha_1 + \gamma1) * z_t \approx - 0.05$ for $z_t = 1$. Observe the graphical representation below.

#### GJR-GARCH(1, 1)

In GJR-GARCH, the asymmetry is driven by the $\gamma_1$ parameter. From

$$ \sigma_t^2 = \omega + (\alpha_1 \epsilon_{t-1}^2 + \gamma_1 I_{t-1} \epsilon_{t-1}^2) + \beta_1 \sigma_{t-1}^2,$$

with $I_t = 1$ if $\epsilon_t \leq 0$, the effect is $\alpha_1$ if $\epsilon_t > 0$, and $(\alpha_1 + \gamma_1)$ if $\epsilon_t \leq 0$.

In [ ]:
effects <- matrix(NA, nrow = 2, ncol = 5)
rownames(effects) <- c('EGARCH-t', 'GJR-GARCH-t')
colnames(effects) <- c('alpha', 'gamma', 'negative', 'positive', 'BIC')

alpha_e <- coef(fit_egarch)['alpha1']
gamma_e <- coef(fit_egarch)['gamma1']
alpha_g <- coef(fit_gjr)['alpha1']
gamma_g <- coef(fit_gjr)['gamma1']

effects['EGARCH-t', ] <- c(alpha_e, gamma_e, -(alpha_e - gamma_e), alpha_e + gamma_e,
                           infocriteria(fit_egarch)[2])

effects['GJR-GARCH-t', ] <- c(alpha_g, gamma_g, alpha_g + gamma_g, alpha_g,
                              infocriteria(fit_gjr)[2])

round(effects, 4)


In [ ]:
res <- seq(-3, 3, 0.01)
plot(res, alpha_e * res + gamma_e * abs(res), type = 'l', lwd = 2,
     main = 'EGARCH shock impact', xlab = 'standardized shock', ylab = 'impact on log-volatility')
abline(h = 0, v = 0, lty = 2, col = 'grey50')

plot(res, (alpha_g * res^2 + gamma_g * ifelse(res < 0, 1, 0)) * res^2, type = 'l', lwd = 2,
     main = 'GJR-GARCH shock impact', xlab = 'shock', ylab = 'impact on variance')
abline(h = 0, v = 0, lty = 2, col = 'grey50')


In [ ]:
IC <- do.call(cbind, list(
  "sGARCH-t"   = infocriteria(fit_standard),
  "eGARCH-t"   = infocriteria(fit_egarch),
  "gjrGARCH-t" = infocriteria(fit_gjr)
))
colnames(IC) <- c('GARCH', 'EGARCH', 'GJR-GARCH')
IC

## Long memory properties in financial time series

### Long memory in ARMA models

Not all processes are "well behaved" enough to be characterized by ARMA models. Lot of processes observable in the real world (e.g. in economics, tellecomunications or meteorology) exhibit correlations that do not decay sufficiently quickly. To avoid misspecification due to problematic distinguishing between I(0) and I(1) models, long memory can be modelled using Fractionally Integrated ARMA models (FARIMA or ARFIMA).

Definition of an ARFIMA(p, d, q) model follows directly from ARIMA(p, d, q):

$$\phi (L) (1 - L)^d X_t = \theta (L) \epsilon_t,$$

where d is a non-negative integer. ARFIMA(p, d, q) is a generalization of the former model that allows d to take **any real value**.

In [ ]:
set.seed(286)
phi1 <- 0.9
ar0 <- arima.sim(model = list(ar = phi1), n = 1000)
ar1 <- vector()
eps <- rnorm(1000)
ar1[1] <- 0
for (t in 2:1000){
    ar1[t] <- ar1[t - 1] + eps[t]
}
par(mfrow = c(1, 2))
plot.ts(ar0)
plot.ts(ar1)
acf(ar0)
acf(ar1)

In [ ]:
set.seed(286)
data1 <- arfima.sim(model = list(phi = numeric(0), theta = numeric(0), dfrac = .3), n = 1000)
plot.ts(data1, ylab = NA, type = 'l', col = 'steelblue', main = 'ARFIMA(0, 0.3, 0)', cex.main = 0.9)

par(mfrow = c(1, 2))
acf(data1, main = "")
pacf(data1, main = "")

In [ ]:
specar <- arfimaspec(mean.model = list(armaOrder = c(1, 0)), fixed.pars = list(ar1 = 0.3, mu = 0, sigma = 1))
specarfima <- arfimaspec(mean.model = list(armaOrder = c(0, 0), arfima = TRUE), 
                         fixed.pars = list(mu = 0, sigma = 1, arfima = 0.3))

In [ ]:
forecast1 <- arfimaforecast(specar, data = data1[1:990], n.ahead = 10)
forecast2 <- arfimaforecast(specarfima, data = data1[1:990], n.ahead = 10)

In [ ]:
plot.ts(data1[901:1000], ylab = NA, type = 'l', col = 'gray70', main = 'ARFIMA(0, 0.3, 0)', cex.main = 0.9)
lines(rep(0, 100), lwd = 0.5)
lines(c(rep(NA, 89), data1[990], forecast1@forecast$seriesFor), col = 'steelblue')
lines(c(rep(NA, 89), data1[990], forecast2@forecast$seriesFor), col = 'darkorange')
legend("bottomright", ncol = 2, legend = c('AR', 'ARFIMA'),col = c('steelblue', 'darkorange'), lwd = 2)

The AR forecast returns to the mean too quickly, while ARFIMA preserves the slowly decaying dependence.

### Approximation of ARFIMA with AR process

An ARFIMA process, can be approximated by an appropriate ARMA process. However, the ARMA approximation is not able to efficiently capture the properties with growing number of observations.

In [ ]:
arf <- arfima.sim(model = list(phi = 0.8, dfrac = 0.4), n = 200)

l <- 205
e <- rnorm(l)
y <- vector()
phi1 <- 1.7
phi2 <- -1.05
phi3 <- 0.493
phi4 <- -0.2561
phi5 <- 0.124722
y[1] <- 0
y[2] <- phi1 * e[1] + e[2]
y[3] <- phi1 * y[2] + e[3]
y[4] <- phi1 * y[3] + phi2 * y[2] + e[4]
y[5] <- phi1 * y[4] + phi2 * y[3] + phi3 * y[2] + e[5]
for (i in 6:l){
    y[i] <- phi1 * y[i - 1] + phi2 * y[i - 2] + phi3 * y[i - 3] + phi4 * y[i - 4] + phi5 * y[i - 5] + e[i]
}


plot.ts(arf, col = 'steelblue', ylim = c(-50, 50))
lines(y, col = 'yellow3')

### Autocorrelation decay

Simulate an ARMA(1, 0) process with $\phi_1 = 0.8$, and the corresponding model with long memory component $d = 0.4$. Compare the decay in ACFs.

In [ ]:
set.seed(286)
arfima <- arfima.sim(model = list(phi =  0.8, theta = numeric(0), dfrac = 0.4), n = 500)
ar <- arfima.sim(model = list(phi = 0.8, theta = numeric(0)), n = 500)


In [ ]:
acf1 <- acf(ar, plot = FALSE,lag.max = 20)
acf1DF <- with(acf1, data.frame(lag, acf))
acf1DF$lag <- as.integer(acf1DF$lag)
acf2 <- acf(arfima, plot = FALSE,lag.max = 20)
acf2DF <- with(acf2, data.frame(lag, acf))
acf2DF$lag <- as.integer(acf2DF$lag)

acf1DF$group <- 'AR'
acf2DF$group <- 'ARFIMA'

dat <- rbind(acf1DF,acf2DF)

ggplot(dat, aes(x = factor(lag), y = acf,fill = factor(group))) + 
  geom_bar(width = 0.3, stat = "identity", position = position_dodge())


## Simulated paths from ARFIMA process

Simulate 50 realizations of an ARFIMA process with zero mean, no ARMA part, and $d = 0.45$.

In [ ]:
n <- 50
series <- lapply(1:n, function (x){y <-arfima.sim(model = list(phi = numeric(0), theta = numeric(0), 
                                                                 dfrac = 0.45), n = 100)
                                  return(y)})
histogram <- hist(unlist(series), breaks = seq(-5, 5, 0.5), plot = FALSE)

layout(matrix(c(0, 0, 1, 2), 2, 2, byrow = TRUE), c(3, 1), c(1, 3), TRUE)

par(mar = c(3, 1, 0, 0))
plot(series[[1]], ylab = NA, ylim = c(min(unlist(series)), max(unlist(series))))
for (i in 2:n){
    lines(series[[i]], col = colors()[3 * i])
}
par(mar = c(3, 0, 0, 0))
barplot(histogram$counts, space = 0, axes = F, col = 'steelblue', horiz = TRUE)

### Power Spectral density

Power spectrum carries information about how much signal is at each frequency. Compare theoretical and empirical power spectra of Gaussian White Noise, AR process, and ARFIMA process with no ARMA component.

In [ ]:
set.seed(304)
wn <- rnorm(500)
arfima <- arfima.sim(model = list(phi = numeric(0), theta = numeric(0), dfrac = 0.45), n = 500)
ar <- arfima.sim(model = list(phi = 0.5, theta = numeric(0)), n = 500)


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 7)
del <- 0.1 # sampling interval
x.spec <- spectrum(wn,log = "no", span = 10, plot = FALSE)


par(mfrow=c(1,3))
plot(x.spec$freq / del, x.spec$spec, xlab = 'frequency', ylab = 'spectral density', type = 'l', 
     main = 'White noise', col = 'gray40', ylim = c(0.5,1.7))
legend("topleft", ncol = 2, legend = c('Empirical', 'Theoretical'), col = 'gray40', lwd = 2, 
           lty = c(1, 2), cex = .9, box.col = 'white')
lines(seq(0, 5, 0.01), rep(1, 5/0.01 + 1), type = 'l', xlab = 'frequency', ylab = NA, 
     main = 'Theoretical power spectrum', col = 'gray40', lty = 2)
box()

y <- seq(0, 0.5, 0.02)
ar.spec <- spectrum(ar, log = "no", span = 10, plot = FALSE)
arf.spec <- spectrum(arfima, log = "no", span = 10, plot = FALSE)

plot(y / del, 1 / (1 - 2 * 0.5 * cos(2 * pi * y) + 0.5^2), type = 'l', xlab = 'frequency', ylab = NA, 
     main = 'Theoretical power spectrum', ylim = c(0, 5), col = 'steelblue')
lines(y / del, (1 / (2 * pi)) * (2 * sin(y / 2))^(-2 * 0.45), type = 'l', ylim = c(0, 4), 
     xlab = 'frequency', ylab = NA, col = 'darkorange')
legend("topright", ncol = 2, legend = c('AR', 'ARFIMA'), col = c('steelblue', 'darkorange'), lwd = 2, 
           lty = c(1, 2), cex = .88, box.col = 'white')
box()

plot(ar.spec$freq / del, ar.spec$spec, xlab = 'frequency', ylab = 'spectral density', type = 'l', 
     main = 'Empirical Power spectrum', col = 'steelblue', ylim = c(0, 5))
lines(arf.spec$freq / del, arf.spec$spec, xlab = 'frequency', ylab = 'spectral density', type = 'l', 
     col = 'darkorange')
legend("topright", ncol = 2, legend = c('AR', 'ARFIMA'), col = c('steelblue', 'darkorange'), lwd = 2, 
           lty = c(1, 2), cex = .88, box.col = 'white')
box()

### FIGARCH

In [ ]:
ftse <- getSymbols('^FTSE', auto.assign = FALSE, from = as.Date('2008-01-01'), to = as.Date('2011-12-31'))
ftse <- diff(log(ftse), na.rm = TRUE)
ftse <- na.omit(ftse[, 'FTSE.Adjusted'])

In [ ]:
plot(index(ftse), ftse, ylab = 'return', xlab = 'Year', main = 'FTSE', type = 'h', lwd = 2)

In [ ]:
par(mfrow = c(1, 2))
acf(ftse)
pacf(ftse)

In [ ]:
mu <- arima(ftse, order = c(0, 0, 0))
mu
arch.test(mu)

In [ ]:
garch11T <- ugarchspec(variance.model=list(model = "sGARCH", garchOrder = c(1, 1)),
                   mean.model = list(armaOrder=c(0, 0)), distribution.model = 'std')

garch11Tfit <- ugarchfit(garch11T, ftse)

garch11Tfit

In [ ]:
egarch11T <- ugarchspec(variance.model=list(model = "eGARCH", garchOrder = c(1, 1)),
                   mean.model = list(armaOrder=c(0, 0)), distribution.model = 'std')

egarch11Tfit <- ugarchfit(egarch11T, ftse)

egarch11Tfit

In [ ]:
igarch11T <- ugarchspec(variance.model=list(model = "iGARCH", garchOrder = c(1, 1)),
                   mean.model = list(armaOrder=c(0, 0)), distribution.model = 'std')

igarch11Tfit <- ugarchfit(igarch11T, ftse)

igarch11Tfit

In [ ]:
figarch11T <- ugarchspec(variance.model=list(model = "fiGARCH", garchOrder = c(1, 1)),
                   mean.model = list(armaOrder=c(0, 0)), distribution.model = 'std')

figarch11Tfit <- ugarchfit(figarch11T, ftse)

figarch11Tfit

In [ ]:
color <- c('gray40', 'steelblue', 'darkorange')
plot(igarch11Tfit@fit$sigma, type = 'l', ylab = 'Volatility', main = 'GARCH models comparison', ylim = c(0.005, 0.06), 
     col = color[1])
lines(egarch11Tfit@fit$sigma, col = color[3])
lines(figarch11Tfit@fit$sigma, col = color[2])

legend("top", ncol = 3, legend = c('iGARCH(1, 1)', 'FIGARCH(1, 1)', 
      'EGARCH(1, 1)'), col = color, lwd = 2, 
       box.col = 'white')

In [ ]:
color <- c('gray40', 'steelblue', 'darkorange')
plot(garch11Tfit@fit$sigma[180:300], type = 'l', ylab = 'Volatility', main = 'GARCH models comparison', ylim = c(0.005, 0.06), 
     col = color[1])
lines(igarch11Tfit@fit$sigma[180:300], col = color[2])
lines(figarch11Tfit@fit$sigma[180:300], col = color[3])

legend("top", ncol = 3, legend = c('GARCH(1, 1)', 'iGARCH(1, 1)', 
      'FIGARCH(1, 1)'), col = color, lwd = 2, 
       box.col = 'white', cex = 1, yjust = 0)

In [ ]:
IC <- do.call(cbind, list(
  "GARCH(1, 1)"   = infocriteria(garch11Tfit),
  "iGARCH(1, 1)"   = infocriteria(igarch11Tfit),
  "eGARCH(1, 1)"   = infocriteria(egarch11Tfit),
  "FIGARCH(1, 1)"   = infocriteria(figarch11Tfit)
))
colnames(IC) <- c("GARCH(1, 1)", "iGARCH(1, 1)", "eGARCH(1, 1)", "FIGARCH(1, 1)")
IC

### Estimation of the long memory parameter

We can use Whittle estimator to estimate the long memory parameter d. Specifically, it estimates parameter H, which is $d + 1/2$. Simulate 100 realizations of ARFIMA process, and see the estimated values of d.

In [ ]:
n <- 100
W1 <- lapply(lapply(1:n, function (x){
    arfima.sim(model = list(phi = numeric(0), theta = numeric(0), dfrac = 0.15), n = 100)}),
    function (y){
        return(WhittleEst(y)$coefficients[1] - 0.5)})

W2 <- lapply(lapply(1:n, function (x){
    arfima.sim(model = list(phi = 0.3, theta = numeric(0), dfrac = 0.15), n = 100)}),
    function (y){
        return(WhittleEst(y)$coefficients[1] - 0.5)})

In [ ]:
par(mfrow = c(1, 2))

hist(unlist(W1), main = 'No AR part', col = 'steelblue', xlab = NA, breaks = 20)
hist(unlist(W2), main = 'ARFIMA(1, 0.15, 0)', col = 'darkorange', xlab = NA, breaks = 20)

Note that the true value of the parameter d is equal to 0.15. When autoregressive component is added, the estimator displays substantial bias. Local Whittle estimator reduces the number of frequencies used in estimation, which reduces the bias. Estimates of d from the ARFIMA (1, 0.15, 0) using classical Whittle and Local Whittle estimator are confirmed below.

In [ ]:
R.lw<-function(d,peri,m,T,l=1){
  lambda<-2*pi/T
  K<-log(1/(m-l-1)*(sum(peri[l:m]*(lambda*(l:m))^(2*d))))-2*d/(m-l-1)*sum(log(lambda*(l:m)))
  K
}

#'Concentrated local Whittle likelihood of differenced and tapered estimator of 
#'Hurvich and Chen (2000). Only for internal use. 
#'@keywords internal
R.lw.hc<-function (d, peri, m, T) 
{
  j <- (1:m) + 1/2
  lambdaj <- 2 * pi *j /T
  K <- log(mean(peri[1:m] * (lambdaj )^(2 * d))) -  2 * d*mean(log(lambdaj ))
  K
}


#'Cosine Bell Taper
#'@keywords internal
cos_bell<-function(u){1/2*(1-cos(2*pi*u))}

#'Complex Cosine Bell Taper
#'@keywords internal
cos_bell_cmplx<-function(u){1/2*(1-cos(1i*2*pi*u))}

#'Concentrated local Whittle likelihood for tapered estimate. Only for internal use. Cf. Velasco (1999).
#'@keywords internal
R.lw.tapered<-function(d,peri,m,p,T){
  lambda<-2*pi/T
  j<-seq(p,m,p)
  K<-log(p/m*(sum(peri[j]*(lambda*(j))^(2*d))))-2*d*p/m*sum(log(lambda*(j)))
  K
}

#'Scaling factor in the asymptotic variance. Only for internal use. Cf. Velasco (1999).
#'@keywords internal
PHI_lw<-function(ht,T,p){
  k<-seq(0,T-p,p)
  lambdak<-2*pi/T*k
  enumerator<-0
  for(i in 1:length(k)){
    enumerator<-enumerator+sum(ht^2*cos((1:T)*lambdak[i]))^2
  }
  denominator<-sum(ht^2)^2
  enumerator/denominator
}

#---------------------------       local whittle estimation       ------------------------------------#

#' @title Local Whittle estimator of fractional difference parameter d.
#' @description \code{local.W} Semiparametric local Whittle estimator for memory parameter d following Robinson (1995).
#'  Returns estimate and asymptotic standard error.
#' @details add details here.
#' @param data vector of length T.
#' @param m bandwith parameter specifying the number of Fourier frequencies.
#' used for the estimation usually \code{floor(1+T^delta)}, where 0<delta<1.
#' @param int admissible range for d. Restricts the interval 
#'        of the numerical optimization.
#' @param taper string that determines whether the standard local Whittle estimator of Robinson (1995), 
#' the tapered version of Velasco (1999), or the differenced and tapered estimator of Hurvich and Chen (2000) is used.
#' @import longmemo
#' @references Robinson, P. M. (1995): Gaussian Semiparametric Estimation of Long Range Dependence. 
#' The Annals of Statistics, Vol. 23, No. 5, pp. 1630 - 1661.
#' Velasco, C. (1999): Gaussian Semiparametric Estimation for Non-Stationary Time Series.
#' Journal of Time Series Analysis, Vol. 20, No. 1, pp. 87-126. 
#' Hurvich, C. M., and Chen, W. W. (2000): An Efficient Taper for Potentially 
#' Overdifferenced Long-Memory Time Series. Journal of Time Series Analysis, Vol. 21, No. 2, pp. 155-180.
#' @examples
#' library(fracdiff)
#' T<-1000
#' d<-0.4
#' series<-fracdiff.sim(n=T, d=d)$series
#' local.W(series,m=floor(1+T^0.65))
#' @export

local.W<-function(data,m,int=c(-0.5,2.5), taper=c("none","Velasco","HC"), diff_param=1, l=1){
  taper<-taper[1]
  if((taper%in%c("none","Velasco","HC"))==FALSE)stop('taper must be either "none", "Velasco", or "HC".')
  T<-length(data)
  if(taper=="Velasco"){
    p<-3
    ht<-cos_bell((1:T)/T)
    data<-ht*data
    peri<-per(data)[-1]
    d.hat<-optimize(f=R.lw.tapered, interval=int, peri=peri,  m=m, T=T, p=p)$minimum
    se<-sqrt(p*PHI_lw(ht=ht,T=T,p=p)/(4*m))
  }
  if(taper=="HC"){
    data<-diff(data,differences=diff_param)
    T<-length(data)
    ht<-cos_bell_cmplx((1:T)/T)
    data<-ht*data
    peri<-per(data)[-1]
    d.hat<-optimize(f=R.lw.hc, interval=int, peri=peri,  m=m, T=T)$minimum+1
    se<-sqrt(1.5/(4*m))
  }
  if(taper=="none"){
    peri<-per(data)[-1]
    d.hat<-optimize(f=R.lw, interval=int, peri=peri,  m=m, T=T, l=1)$minimum
    se<-1/(2*sqrt(m))
  }
  return(list("d"=d.hat, "s.e."=se))
}


# library(fracdiff)
# T<-1000
# d<-0.4
# series<-fracdiff.sim(n=T, d=d)$series
# local.W(series,m=floor(1+T^0.65))

In [ ]:
series <- lapply(1:n, function (x){
    arfima.sim(model = list(phi = 0.3, theta = numeric(0), dfrac = 0.15), n = 100)})

W3 <- lapply(series, function (y){
        return(local.W(y, m = floor(1 + length(y)^0.65))$d)})

W4 <- lapply(series, function (y){
        return(WhittleEst(y)$coefficients[1] - 0.5)})

par(mfrow = c(1, 2))

hist(unlist(W3), main = 'Local Whittle', col = 'steelblue', xlab = NA, breaks = 20)
hist(unlist(W4), main = 'Whittle', col = 'darkorange', xlab = NA, breaks = 20)

Reduction in bias is offset by larger variance of the estimates.